# Paper demo: GPT-2 toy example

This notebook reproduces the paper's toy experiment on **all 50,256 non-special GPT-2 tokens at layer 0**. It captures the real residual-stream activations, fits full-width ICA for 50 iterations, and compares a seeded random projection with the recovered component whose tail groups *neither / either / both / between* tokens.

A T4 runtime is sufficient. Runtime depends on model download and GPU availability.

In [ ]:
%pip install -q icalens==0.4.0 scipy matplotlib transformers

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import kurtosis, norm
from transformers import AutoModelForCausalLM, AutoTokenizer
from icalens import ICALens

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then run again."
model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).cuda().eval()
token_ids = [i for i in range(len(tokenizer)) if i not in set(tokenizer.all_special_ids)]
chunks = []
positions = torch.arange(2, device="cuda")
with torch.inference_mode():
    for ids in torch.tensor(token_ids).split(1024):
        batch = torch.stack((torch.full_like(ids, tokenizer.eos_token_id), ids), dim=1).cuda()
        hidden = model.transformer.wte(batch) + model.transformer.wpe(positions)
        hidden = model.transformer.drop(hidden)
        output = model.transformer.h[0](hidden, use_cache=False)
        hidden = output[0] if isinstance(output, tuple) else output
        chunks.append(hidden[:, 1].detach().float().cpu())
activations = torch.cat(chunks)
print("Captured activations:", tuple(activations.shape))

In [ ]:
lens = ICALens(model_id=model_id, model_type="base", activation_site="resid_post")
lens.fit(activations, layer=0, max_iter=50, batch_size=8192, device="cuda", random_state=0, progress=True)
scores = torch.as_tensor(lens.transform(activations, layer=0)).float()
component_kurtosis = np.asarray(kurtosis(scores.numpy(), axis=0))

families = {" neither", " Neither", "Neither", " either", " Either", "Either", " both", " Both", "Both", " between", " Between", "Between"}
decoded = [tokenizer.decode([i]) for i in token_ids]
family_rows = [i for i, token in enumerate(decoded) if token in families]
quality = []
for component in range(scores.shape[1]):
    orientation = 1 if scores[family_rows, component].mean() >= 0 else -1
    top = torch.topk(orientation * scores[:, component], 20).indices.tolist()
    overlap = len(set(top) & set(family_rows))
    quality.append((overlap, component_kurtosis[component], component, orientation, top))
overlap, excess, component, orientation, top = max(quality)
print(f"Selected C{component}: {overlap} family tokens in top 20; excess kurtosis {excess:.1f}")
print([decoded[i] for i in top])

In [ ]:
generator = torch.Generator().manual_seed(0)
centered = activations - activations.mean(0)
random_direction = torch.randn(centered.shape[1], generator=generator)
random_direction /= random_direction.norm()
random_scores = centered @ random_direction
ica_scores = orientation * scores[:, component]

x = ((random_scores-random_scores.mean())/random_scores.std()).numpy()
y = ((ica_scores-ica_scores.mean())/ica_scores.std()).numpy()
background = torch.randperm(len(token_ids), generator=generator)[:500].numpy()
related = np.asarray(top)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(x[background], y[background], s=14, color="#aeb8c2", alpha=.55, label="Sampled GPT-2 tokens")
ax.scatter(x[related], y[related], s=34, color="#3568a8", label="ICA-tail token family", zorder=3)
for index in related[:10]:
    ax.annotate(decoded[index].replace(" ", "␣"), (x[index], y[index]), xytext=(4,3), textcoords="offset points", fontsize=8)
ax.annotate("", xy=(3.2,0), xytext=(0,0), arrowprops={"arrowstyle":"->","lw":2.2,"color":"#4f8f78"})
ax.annotate("random direction w", xy=(3.2,0), xytext=(8,7), textcoords="offset points", color="#356b59", fontweight="bold")
ax.annotate("", xy=(0,9), xytext=(0,0), arrowprops={"arrowstyle":"->","lw":2.2,"color":"#3568a8"})
ax.annotate(f"ICA direction v (C{component})", xy=(0,9), xytext=(8,-5), textcoords="offset points", color="#3568a8", fontweight="bold")
ax.axhline(0,color="#d8dedf",lw=.8); ax.axvline(0,color="#d8dedf",lw=.8)
ax.set_xlim(-4.5,4.5); ax.set_ylim(-4,14)
ax.set_xlabel("Projection onto random direction w")
ax.set_ylabel("Projection onto ICA direction v")
ax.set_title("GPT-2 layer-0 token activations in the two-direction plane", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="upper left")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

This is the Figure 1(a) view: the same activation population is shown in a two-direction plane. The random direction mixes the vocabulary, while the ICA direction aligns a coherent token family into a selective tail.